# Clustering

This notebook loads ACE user embeddings from a saved experiment and runs UMAP + HDBSCAN clustering. It is intentionally downstream of `02_ace.ipynb`; it should not retrain ACE.


In [1]:
import os
import json
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import umap
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from joblib import Parallel, delayed

In [ ]:
# ================================
# Experiment Selection
# ================================

city = 'london'
loc_embedding_type = 'aether'
save_dir = f'pretrained_ace/{city}_{loc_embedding_type}'

# Set this to the ACE experiment folder you created and trained the model in previously.
# Example: '20260523_183407_mask-location_only_emb-prompt_prompt-1_maskp0.2_minmask0_sw1_mw1'
# experiment_name = '20260525_002940_mask-all_emb-prompt_prompt-1_maskp0.2_minmask0_sw1_mw1'
experiment_name = '20260525_180711_mask-all_emb-prompt_prompt-1_maskp0.2_minmask0_sw1_mw1'
# experiment_name = 'original'

# Choose which ACE representation to cluster: {'overall', 'weekday', 'weekend'}.
clustering_embedding_type = 'weekend'
fallback_to_overall_embedding = False

# UMAP/HDBSCAN parameters.
umap_2d_n_neighbors = 30
umap_2d_min_dist = 0.0
umap_n_components = 5
umap_n_neighbors = 30
umap_min_dist = 0.0
umap_metric = 'cosine'
umap_random_state = 101

hdbscan_min_cluster_size = 1000
hdbscan_min_samples = 30


In [3]:
# ================================
# Experiment Path Helper
# ================================

def make_ace_experiment_paths(save_dir, city, loc_embedding_type, clustering_embedding_type, experiment_name, create_dirs=True, update_globals=True):
    """Build and optionally create the standard ACE experiment artifact paths."""
    paths = {
        'experiment_name': experiment_name,
        'experiment_dir': os.path.join(save_dir, 'experiments', experiment_name),
    }
    paths.update({
        'model_dir': os.path.join(paths['experiment_dir'], 'models'),
        'log_dir': os.path.join(paths['experiment_dir'], 'logs'),
        'embedding_dir': os.path.join(paths['experiment_dir'], 'embeddings'),
        'clustering_dir': os.path.join(paths['experiment_dir'], 'clustering'),
        'figure_dir': os.path.join(paths['experiment_dir'], 'figures'),
    })
    paths.update({
        'save_path': os.path.join(paths['model_dir'], f'best_ace_model_{city}_{loc_embedding_type}.pt'),
        'training_loss_plot_path': os.path.join(paths['figure_dir'], 'training_losses.png'),
        'training_history_path': os.path.join(paths['log_dir'], 'training_history.csv'),
        'experiment_config_path': os.path.join(paths['log_dir'], 'experiment_config.json'),
        'embedding_save_path': os.path.join(paths['embedding_dir'], f'user_embeddings_{city}_{loc_embedding_type}.pt'),
        'chain_embedding_save_path': os.path.join(paths['embedding_dir'], f'chain_embeddings_{city}_{loc_embedding_type}.pt'),
        'metadata_save_path': os.path.join(paths['embedding_dir'], 'user_embeddings_metadata.pt'),
        'clustering_results_path': os.path.join(paths['clustering_dir'], f'user_clustering_results_{city}_{loc_embedding_type}_{clustering_embedding_type}.csv'),
    })
    if create_dirs:
        for dir_key in ['experiment_dir', 'model_dir', 'log_dir', 'embedding_dir', 'clustering_dir', 'figure_dir']:
            os.makedirs(paths[dir_key], exist_ok=True)
    if update_globals:
        globals().update(paths)
    return paths


def list_ace_experiments(save_dir):
    experiments_dir = os.path.join(save_dir, 'experiments')
    if not os.path.isdir(experiments_dir):
        return []
    return sorted(
        name for name in os.listdir(experiments_dir)
        if os.path.isdir(os.path.join(experiments_dir, name))
    )


In [4]:
if not experiment_name:
    available_experiments = list_ace_experiments(save_dir)
    print('Available ACE experiments:')
    for name in available_experiments[-10:]:
        print('  ', name)
    raise ValueError('Set `experiment_name` to one of the saved ACE experiment folders before continuing.')

experiment_paths = make_ace_experiment_paths(save_dir, city, loc_embedding_type, clustering_embedding_type, experiment_name)
print(f"Clustering experiment directory: {experiment_paths['clustering_dir']}")


Clustering experiment directory: pretrained_ace/london_aether/experiments/20260525_180711_mask-all_emb-prompt_prompt-1_maskp0.2_minmask0_sw1_mw1/clustering


In [5]:
# ================================
# Load ACE Embeddings and Metadata
# ================================

if not os.path.exists(embedding_save_path):
    raise FileNotFoundError(f'User embedding file not found: {embedding_save_path}')

user_embeddings = torch.load(embedding_save_path, map_location='cpu')
metadata = torch.load(metadata_save_path, map_location='cpu') if os.path.exists(metadata_save_path) else {}

user_id_mapping_path = os.path.join(experiment_dir, 'user_id_mapping.csv')
if not os.path.exists(user_id_mapping_path):
    raise FileNotFoundError(f'User ID mapping file not found: {user_id_mapping_path}')
df_user_id_mapping = pd.read_csv(user_id_mapping_path)

print(f"Loaded {len(user_embeddings)} user embedding records")
print(f"Embedding metadata keys: {list(metadata.keys())}")


Loaded 112112 user embedding records
Embedding metadata keys: ['num_users', 'embedding_dim', 'embedding_fields', 'num_users_with_weekday_embedding', 'num_users_with_weekend_embedding', 'user_ids', 'experiment_name', 'experiment_dir', 'model_config', 'best_model_path']


In [6]:
# ================================
# Select Embedding Representation
# ================================

embedding_field_map = {
    'overall': 'overall_embedding',
    'weekday': 'weekday_embedding',
    'weekend': 'weekend_embedding',
}
if clustering_embedding_type not in embedding_field_map:
    raise ValueError(f"clustering_embedding_type must be one of {set(embedding_field_map)}, got {clustering_embedding_type}")

selected_embedding_field = embedding_field_map[clustering_embedding_type]
user_embedding_rows = []
for uid, embedding_record in user_embeddings.items():
    # Backward compatibility: older saved files stored uid -> tensor.
    if isinstance(embedding_record, torch.Tensor):
        selected_embedding = embedding_record
        used_fallback = False
        n_total_chains = np.nan
        n_weekday_chains = np.nan
        n_weekend_chains = np.nan
    else:
        selected_embedding = embedding_record[selected_embedding_field]
        used_fallback = False
        if selected_embedding is None and fallback_to_overall_embedding:
            selected_embedding = embedding_record['overall_embedding']
            used_fallback = True
        n_total_chains = embedding_record['n_total_chains']
        n_weekday_chains = embedding_record['n_weekday_chains']
        n_weekend_chains = embedding_record['n_weekend_chains']

    if selected_embedding is None:
        continue

    user_embedding_rows.append({
        'user_id_num': uid,
        'embedding': selected_embedding.detach().cpu().numpy(),
        'embedding_type': clustering_embedding_type,
        'embedding_field': selected_embedding_field,
        'used_overall_fallback': used_fallback,
        'n_total_chains': n_total_chains,
        'n_weekday_chains': n_weekday_chains,
        'n_weekend_chains': n_weekend_chains,
    })

df_user_clustering = pd.DataFrame(user_embedding_rows)
df_user_clustering = pd.merge(df_user_clustering, df_user_id_mapping, on='user_id_num', how='left')
df_user_clustering = df_user_clustering.sort_values(by='user_id_num').reset_index(drop=True)
user_embedding_values = df_user_clustering['embedding'].values.tolist()
ape_matrix = np.array(user_embedding_values)

df_user_clustering.drop(columns=['embedding'], inplace=True)

print(f"Prepared {len(df_user_clustering)} users for {clustering_embedding_type} clustering")
print(f"Users using overall fallback: {df_user_clustering['used_overall_fallback'].sum()}")
print(f"APE matrix shape: {ape_matrix.shape}")
df_user_clustering


Prepared 83093 users for weekend clustering
Users using overall fallback: 0
APE matrix shape: (83093, 128)


,user_id_num,embedding_type,embedding_field,used_overall_fallback,n_total_chains,n_weekday_chains,n_weekend_chains,user_id
0,1,weekend,weekend_embedding,False,1,0,1,0000c7af-c002-4e39-ad0e-975c01dbe25e
1,2,weekend,weekend_embedding,False,5,4,1,000135bd-bf0b-4880-af99-1791ce98dddf
2,3,weekend,weekend_embedding,False,14,10,4,00016f95-edc9-4642-a6e6-70ce63b66346
3,4,weekend,weekend_embedding,False,14,10,4,0001dfff-15bc-404b-a5f2-a1ec0b458766
4,5,weekend,weekend_embedding,False,1,0,1,0001e685-f381-4bb5-ace6-2949553f06e6
...,...,...,...,...,...,...,...,...
83088,112105,weekend,weekend_embedding,False,12,8,4,fffc7c43-0ab0-40cb-afad-93672cdfc9f5
83089,112106,weekend,weekend_embedding,False,9,7,2,fffdfff3-7c45-43c0-a23e-8534c72a3a19
83090,112107,weekend,weekend_embedding,False,3,2,1,fffeb333-944f-4663-aea6-f310fe721d03
83091,112108,weekend,weekend_embedding,False,8,6,2,ffff1696-ce5a-499c-a567-1a9f612ca0c2


## UMAP Projection


In [ ]:
# 2D UMAP is used for visualization.
reducer2d = umap.UMAP(
    n_neighbors=umap_2d_n_neighbors,
    min_dist=umap_2d_min_dist,
    n_components=2,
    metric=umap_metric,
    random_state=umap_random_state,
)
embedding_2d = reducer2d.fit_transform(ape_matrix)
df_user_clustering['umap_x'] = embedding_2d[:, 0]
df_user_clustering['umap_y'] = embedding_2d[:, 1]
print('2D UMAP complete:', embedding_2d.shape)

plt.figure(figsize=(10, 6))
plt.scatter(df_user_clustering['umap_x'], df_user_clustering['umap_y'], s=5, alpha=0.5)
plt.title(f"UMAP Projection of ACE User Embeddings ({clustering_embedding_type})")
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
umap_2d_plot_path = os.path.join(figure_dir, f'umap_2d_{clustering_embedding_type}.png')
plt.savefig(umap_2d_plot_path, dpi=300, bbox_inches='tight')
print(f'2D UMAP plot saved to {umap_2d_plot_path}')
plt.show()

In [7]:
umap_embedding_path = os.path.join(
    clustering_dir,
    f'embedding_{clustering_embedding_type}_{umap_n_components}d_{city}_{loc_embedding_type}.npz'
)

# nD UMAP is used for HDBSCAN clustering.
if os.path.exists(umap_embedding_path):
    embedding = np.load(umap_embedding_path)['my_embedding']
    print(f'{umap_n_components}D UMAP embedding loaded from {umap_embedding_path}')
else:
    reducer = umap.UMAP(
        n_neighbors=umap_n_neighbors,
        min_dist=umap_min_dist,
        n_components=umap_n_components,
        metric=umap_metric,
        random_state=umap_random_state,
    )
    embedding = reducer.fit_transform(ape_matrix)
    np.savez_compressed(umap_embedding_path, my_embedding=embedding)
    print(f'{umap_n_components}D UMAP embedding saved to {umap_embedding_path}')
print('nD UMAP complete:', embedding.shape)


5D UMAP embedding loaded from pretrained_ace/london_aether/experiments/20260525_180711_mask-all_emb-prompt_prompt-1_maskp0.2_minmask0_sw1_mw1/clustering/embedding_weekend_5d_london_aether.npz
nD UMAP complete: (83093, 5)


In [ ]:
# nD UMAP complete: (104711, 5)

## HDBSCAN Clustering


In [ ]:
# hdbscan_min_cluster_size = 1000
# hdbscan_min_samples = 50

In [ ]:
import os
import gc
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Full trees for ~100k users are slow to render, so keep the plot simplified.
fixed_cluster_size = 1000
condensed_tree_min_samples_list = [30, 50, 100, 200, 300, 500, 1000]
max_rectangles_per_icicle = 5

def plot_hdbscan_condensed_tree(min_samples, min_cluster_size=fixed_cluster_size):
    print(f"Fitting HDBSCAN: min_cluster_size={min_cluster_size}, min_samples={min_samples}...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
    ).fit(embedding)

    labels = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = np.count_nonzero(labels == -1)
    proportion_outliers = n_outliers / len(labels)
    print(f"{n_clusters} clusters, {n_outliers} outliers ({proportion_outliers:.2%})")

    fig, ax = plt.subplots(figsize=(12, 6))
    clusterer.condensed_tree_.plot(
        select_clusters=False,  # avoids a hdbscan/matplotlib ellipse-rendering bug
        axis=ax,
        colorbar=False,
        cmap="viridis",
        max_rectangles_per_icicle=max_rectangles_per_icicle,
    )
    ax.set_title(
        f"Condensed tree: min_cluster_size={min_cluster_size}, min_samples={min_samples}"
    )
    plt.tight_layout()

    condensed_tree_path = os.path.join(
        figure_dir,
        f"hdbscan_condensed_tree_size{min_cluster_size}_sample{min_samples}.png",
    )
    fig.savefig(condensed_tree_path, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved condensed tree to {condensed_tree_path}")

    return {
        "min_cluster_size": min_cluster_size,
        "min_samples": min_samples,
        "num_clusters": n_clusters,
        "num_outliers": n_outliers,
        "proportion_outliers": proportion_outliers,
        "mean_cluster_persistence": (
            float(np.mean(clusterer.cluster_persistence_))
            if len(clusterer.cluster_persistence_) else np.nan
        ),
        "condensed_tree_path": condensed_tree_path,
    }

condensed_tree_rows = []
for ms in condensed_tree_min_samples_list:
    condensed_tree_rows.append(plot_hdbscan_condensed_tree(ms))
    gc.collect()

df_condensed_tree_summary = pd.DataFrame(condensed_tree_rows)
df_condensed_tree_summary


In [8]:
hdbscan_params_combinations = [
    {'min_cluster_size': 1000, 'min_samples': 30},
    {'min_cluster_size': 1000, 'min_samples': 50},
    {'min_cluster_size': 1000, 'min_samples': 100},
    {'min_cluster_size': 1000, 'min_samples': 200},
    {'min_cluster_size': 1000, 'min_samples': 300},
    {'min_cluster_size': 1000, 'min_samples': 500},
    {'min_cluster_size': 1000, 'min_samples': 1000},
    {'min_cluster_size': 2000, 'min_samples': 30},
    {'min_cluster_size': 2000, 'min_samples': 50},
    {'min_cluster_size': 2000, 'min_samples': 100},
    {'min_cluster_size': 2000, 'min_samples': 200},
    {'min_cluster_size': 2000, 'min_samples': 500},
    {'min_cluster_size': 2000, 'min_samples': 1000},
    {'min_cluster_size': 2000, 'min_samples': 2000}
]
hdbscan_params_combinations

[{'min_cluster_size': 1000, 'min_samples': 30},
 {'min_cluster_size': 1000, 'min_samples': 50},
 {'min_cluster_size': 1000, 'min_samples': 100},
 {'min_cluster_size': 1000, 'min_samples': 200},
 {'min_cluster_size': 1000, 'min_samples': 300},
 {'min_cluster_size': 1000, 'min_samples': 500},
 {'min_cluster_size': 1000, 'min_samples': 1000},
 {'min_cluster_size': 2000, 'min_samples': 30},
 {'min_cluster_size': 2000, 'min_samples': 50},
 {'min_cluster_size': 2000, 'min_samples': 100},
 {'min_cluster_size': 2000, 'min_samples': 200},
 {'min_cluster_size': 2000, 'min_samples': 500},
 {'min_cluster_size': 2000, 'min_samples': 1000},
 {'min_cluster_size': 2000, 'min_samples': 2000}]

### Evaluate HDBSCAN parameter combinations with DBCV


In [9]:
import os
import gc
import numpy as np
import pandas as pd
import hdbscan as hdbscan_pkg
from hdbscan.validity import validity_index

# DBCV is higher when clusters are denser and better separated.
# It ranges roughly from -1 to 1; larger is better.
dbcv_params_combinations = hdbscan_params_combinations
dbcv_metric = "euclidean"
dbcv_core_dist_n_jobs = 6

# Exact full-data DBCV can be slow for ~100k users.
# Use None for exact DBCV, or an integer such as 30000 for a faster sampled score.
# dbcv_sample_size = 30000
dbcv_sample_size = None
dbcv_random_state = umap_random_state

def _make_dbcv_score_subset(labels, sample_size=dbcv_sample_size, random_state=dbcv_random_state):
    if sample_size is None or sample_size >= len(labels):
        return np.arange(len(labels))
    rng = np.random.default_rng(random_state)
    return np.sort(rng.choice(len(labels), size=sample_size, replace=False))

def evaluate_hdbscan_dbcv(params):
    min_cluster_size = params["min_cluster_size"]
    min_samples = params["min_samples"]
    print(f"Evaluating min_cluster_size={min_cluster_size}, min_samples={min_samples}...")

    clusterer = hdbscan_pkg.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=dbcv_metric,
        core_dist_n_jobs=dbcv_core_dist_n_jobs,
    ).fit(embedding)

    labels = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = np.count_nonzero(labels == -1)
    proportion_outliers = n_outliers / len(labels)

    if n_clusters < 2:
        dbcv_score = np.nan
        dbcv_error = "DBCV requires at least two non-noise clusters"
        score_n_users = 0
    else:
        score_idx = _make_dbcv_score_subset(labels)
        score_embedding = np.asarray(embedding[score_idx], dtype=np.float64)
        score_labels = labels[score_idx]
        score_n_clusters = len(set(score_labels)) - (1 if -1 in score_labels else 0)
        score_n_users = len(score_idx)
        if score_n_clusters < 2:
            dbcv_score = np.nan
            dbcv_error = "DBCV sample has fewer than two non-noise clusters"
        else:
            try:
                dbcv_score = validity_index(score_embedding, score_labels, metric=dbcv_metric)
                dbcv_error = None
            except Exception as exc:
                dbcv_score = np.nan
                dbcv_error = repr(exc)

    mean_cluster_persistence = (
        float(np.mean(clusterer.cluster_persistence_))
        if len(clusterer.cluster_persistence_) else np.nan
    )

    result = {
        "min_cluster_size": min_cluster_size,
        "min_samples": min_samples,
        "num_clusters": n_clusters,
        "num_outliers": n_outliers,
        "proportion_outliers": proportion_outliers,
        "dbcv_score": dbcv_score,
        "dbcv_sample_size": dbcv_sample_size,
        "dbcv_scored_users": score_n_users,
        "mean_cluster_persistence": mean_cluster_persistence,
        "dbcv_error": dbcv_error,
    }
    print(
        f"DBCV={dbcv_score:.4f} | clusters={n_clusters} | "
        f"outliers={n_outliers} ({proportion_outliers:.2%})"
        if np.isfinite(dbcv_score)
        else f"DBCV=nan | clusters={n_clusters} | error={dbcv_error}"
    )
    return result

dbcv_rows = []
for params in dbcv_params_combinations:
    dbcv_rows.append(evaluate_hdbscan_dbcv(params))
    gc.collect()

df_hdbscan_dbcv = pd.DataFrame(dbcv_rows).sort_values(
    by=["dbcv_score", "proportion_outliers"],
    ascending=[False, True],
)

dbcv_summary_path = os.path.join(
    clustering_dir,
    f"hdbscan_dbcv_summary_{clustering_embedding_type}_umap{umap_n_components}d.csv",
)
df_hdbscan_dbcv.to_csv(dbcv_summary_path, index=False)
print(f"DBCV summary saved to {dbcv_summary_path}")
df_hdbscan_dbcv


Evaluating min_cluster_size=1000, min_samples=30...
DBCV=-0.1639 | clusters=7 | outliers=27179 (32.71%)
Evaluating min_cluster_size=1000, min_samples=50...


: 

In [ ]:
additional_hdbscan_params_combinations = [
    {'min_cluster_size': 2000, 'min_samples': 200},
]
additional_hdbscan_params_combinations

In [ ]:
print("Testing HDBSCAN with different parameter combinations:")

# Use all available CPU cores by default. Set this to a smaller integer if memory usage is high.
n_hdbscan_jobs = 6

def run_hdbscan_for_params(params):
    hdbscan_min_cluster_size = params["min_cluster_size"]
    hdbscan_min_samples = params["min_samples"]

    clusterer = HDBSCAN(
        min_cluster_size=hdbscan_min_cluster_size,
        min_samples=hdbscan_min_samples,
    )
    cluster_labels = clusterer.fit_predict(embedding)

    cluster_col = f"clst_{clustering_embedding_type}_umap{umap_n_components}d_size{hdbscan_min_cluster_size}_sample{hdbscan_min_samples}"
    num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    num_outliers = np.count_nonzero(cluster_labels == -1)
    proportion_outliers = num_outliers / len(cluster_labels)

    return {
        "cluster_col": cluster_col,
        "cluster_labels": cluster_labels,
        "min_cluster_size": hdbscan_min_cluster_size,
        "min_samples": hdbscan_min_samples,
        "num_clusters": num_clusters,
        "num_outliers": num_outliers,
        "proportion_outliers": proportion_outliers,
    }

with Parallel(n_jobs=n_hdbscan_jobs) as parallel:
    hdbscan_results = parallel(
        delayed(run_hdbscan_for_params)(params)
        for params in additional_hdbscan_params_combinations
    )

for result in hdbscan_results:
    df_user_clustering[result["cluster_col"]] = result["cluster_labels"]
    print(f"{result['min_cluster_size']}_{result['min_samples']}: {result['num_clusters']} clusters, {result['num_outliers']} ({result['proportion_outliers']:.2%}) outlier users")

del hdbscan_results
gc.collect()
# df_user_clustering[cluster_col].value_counts().sort_index()


# 1500_30, 1500_50, 1000_20 all yields 13 clusters
# 1000_20 yields (13 clusters, 42888 outlier)
# 1000_30 yields (14 clusters, 44910 outlier users) -- and this setting was finally picked! 30 is preferred because it is more aligned with the neighborhood size used in UMAP, and it results in almost the same number of outliers compared to 20.
# 1000_100 yields (13 clusters, 41625 outlier)
# 1000_50 yields (13 clusters, 42441 outlier)
# 2000_30 yields 3 clusters -- definitely not good -- so we have to set min_cluster_size some value between 1000 and 1500.

In [ ]:
df_user_clustering

In [ ]:
df_user_clustering.columns

### Choose the final param combination

The criteria: We would like the number of clusters to be reasonable: 4 to 12

Proportion of outliers: as low as possible.

In [ ]:
final_params = {'min_cluster_size': 2000, 'min_samples': 200}

final_retained_col = f"clst_{clustering_embedding_type}_umap{umap_n_components}d_size{final_params['min_cluster_size']}_sample{final_params['min_samples']}"

df_user_clustering = df_user_clustering[['user_id_num', 'user_id', 'umap_x', 'umap_y', final_retained_col]]
df_user_clustering

In [ ]:
num_clusters = df_user_clustering[final_retained_col].nunique() - 1
num_outliers = (df_user_clustering[final_retained_col] == -1).sum()
cluster_col = final_retained_col

In [ ]:
print(num_clusters, num_outliers)

In [ ]:
# Save clustering outputs and run metadata.
df_user_clustering.to_csv(clustering_results_path, index=False)

clustering_metadata = {
    'city': city,
    'loc_embedding_type': loc_embedding_type,
    'experiment_name': experiment_name,
    'clustering_embedding_type': clustering_embedding_type,
    'fallback_to_overall_embedding': fallback_to_overall_embedding,
    'selected_embedding_field': selected_embedding_field,
    'num_users_clustered': int(len(df_user_clustering)),
    'num_clusters': int(num_clusters),
    'num_outliers': int(num_outliers),
    'cluster_col': cluster_col,
    'umap_2d_n_neighbors': umap_2d_n_neighbors,
    'umap_2d_min_dist': umap_2d_min_dist,
    'umap_n_components': umap_n_components,
    'umap_n_neighbors': umap_n_neighbors,
    'umap_min_dist': umap_min_dist,
    'umap_metric': umap_metric,
    'umap_random_state': umap_random_state,
    'hdbscan_min_cluster_size': final_params['min_cluster_size'],
    'hdbscan_min_samples': final_params['min_samples'],
    'clustering_results_path': clustering_results_path,
    'umap_embedding_path': umap_embedding_path,
}
clustering_metadata_path = os.path.join(
    clustering_dir,
    f'clustering_metadata_{clustering_embedding_type}.json'
)
with open(clustering_metadata_path, 'w') as f:
    json.dump(clustering_metadata, f, indent=2)

print(f'Clustering results saved to {clustering_results_path}')
print(f'Clustering metadata saved to {clustering_metadata_path}')


In [ ]:
# Cluster scatter plot.
plt.figure(figsize=(10, 6))
outliers = df_user_clustering[df_user_clustering[cluster_col] == -1]
plt.scatter(outliers['umap_x'], outliers['umap_y'], c='lightgray', s=1, alpha=0.5, label='Noise / Outliers')

clustered = df_user_clustering[df_user_clustering[cluster_col] != -1]
unique_clusters = np.sort(clustered[cluster_col].unique())
discrete_cmap = plt.get_cmap('Spectral', max(len(unique_clusters), 1))
scatter = plt.scatter(
    clustered['umap_x'],
    clustered['umap_y'],
    c=clustered[cluster_col],
    cmap=discrete_cmap,
    s=2,
    alpha=0.8,
)

plt.suptitle(f"ACE User Clusters ({clustering_embedding_type})", fontsize=14)
plt.title(f"UMAP {umap_n_components}D + HDBSCAN: {num_clusters} clusters, {num_outliers} outliers", fontsize=10)
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.colorbar(scatter, label='Cluster ID')
plt.legend()
plt.tight_layout()
cluster_plot_path = os.path.join(figure_dir, f'clusters_{cluster_col}.png')
plt.savefig(cluster_plot_path, dpi=300, bbox_inches='tight')
print(f'Cluster plot saved to {cluster_plot_path}')
plt.show()


In [ ]:
# Cluster size distribution.
cluster_counts = df_user_clustering[cluster_col].value_counts().sort_index()
plt.figure(figsize=(10, 6))
cluster_counts.plot(kind='bar')
plt.title(f'Cluster Size Distribution ({cluster_col})')
plt.xlabel('Cluster ID')
plt.ylabel('Number of Users')
plt.grid(axis='y', alpha=0.3)
for index, value in enumerate(cluster_counts):
    plt.text(index, value, str(value), ha='center', va='bottom')
plt.tight_layout()
cluster_size_plot_path = os.path.join(figure_dir, f'cluster_sizes_{cluster_col}.png')
plt.savefig(cluster_size_plot_path, dpi=300, bbox_inches='tight')
print(f'Cluster size plot saved to {cluster_size_plot_path}')
plt.show()


## Representative Users


In [ ]:
# Select representative users by cosine similarity to each cluster centroid in the selected ACE space.
representative_top_k = 10
embedding_lookup = {
    uid: emb for uid, emb in zip(df_user_clustering['user_id_num'], user_embedding_values)
}

df_user_embed_clustering = df_user_clustering[['user_id', 'user_id_num', cluster_col]].copy()
df_user_embed_clustering['embedding'] = df_user_embed_clustering['user_id_num'].map(embedding_lookup)

valid_clusters = df_user_embed_clustering[df_user_embed_clustering[cluster_col] != -1].copy()
cluster_centroids = valid_clusters.groupby(cluster_col)['embedding'].apply(lambda x: np.mean(np.stack(x.values), axis=0))

representative_rows = []
for cluster_id, centroid in cluster_centroids.items():
    cluster_users = valid_clusters[valid_clusters[cluster_col] == cluster_id].copy()
    embeddings = np.stack(cluster_users['embedding'].values)
    similarities = cosine_similarity(embeddings, centroid.reshape(1, -1)).flatten()
    top_indices = similarities.argsort()[::-1][:representative_top_k]
    for rank, row_idx in enumerate(top_indices, start=1):
        row = cluster_users.iloc[row_idx]
        representative_rows.append({
            'cluster_id': cluster_id,
            'rank': rank,
            'user_id_num': row['user_id_num'],
            'user_id': row['user_id'],
            'cosine_similarity_to_centroid': float(similarities[row_idx]),
        })

df_representative_users = pd.DataFrame(representative_rows)
representative_users_path = os.path.join(clustering_dir, f'representative_users_{cluster_col}.csv')
df_representative_users.to_csv(representative_users_path, index=False)
print(f'Representative users saved to {representative_users_path}')
df_representative_users.head(20)
